In [1]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,027 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.2 MB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [52.1 kB]
Get:13 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packa

In [2]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    // Base64 encode code payload to bypass all quote & newline syntax errors
    String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));

    // Guarantee marker printing using try...finally and display trailing expressions using ast
    String payload = String.format(
        "import base64, ast, traceback\n" +
        "try:\n" +
        "    code_str = base64.b64decode('%s').decode('utf-8')\n" +
        "    tree = ast.parse(code_str)\n" +
        "    if tree.body and isinstance(tree.body[-1], ast.Expr):\n" +
        "        exec(compile(ast.Module(body=tree.body[:-1], type_ignores=[]), '<string>', 'exec'), globals())\n" +
        "        res = eval(compile(ast.Expression(body=tree.body[-1].value), '<string>', 'eval'), globals())\n" +
        "        if res is not None:\n" +
        "            print(res)\n" +
        "    else:\n" +
        "        exec(compile(tree, '<string>', 'exec'), globals())\n" +
        "except Exception:\n" +
        "    traceback.print_exc()\n" +
        "finally:\n" +
        "    print('%s', flush=True)\n",
        b64, marker
    );

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}

import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.PrintStream;
import java.io.IOException;

Process pythonProcess;
PrintStream pyIn;
BufferedReader pyOut;

try {
    ProcessBuilder pb = new ProcessBuilder("python3", "-i", "-u");
    // Redirect stderr to stdout to prevent pipe deadlock and capture Python errors
    pb.redirectErrorStream(true);
    pythonProcess = pb.start();

    pyIn = new PrintStream(pythonProcess.getOutputStream(), true);
    pyOut = new BufferedReader(new InputStreamReader(pythonProcess.getInputStream()));

    System.out.println("Python interpreter initialized safely.");
} catch (IOException e) {
    e.printStackTrace();
}

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Sep 28, 2026 9:58:00 AM java.util.prefs.FileSystemPreferences$1 run
INFO: Created user preferences directory.
->  void runPy(String code) throws Exception {
>>          String marker = "___END_OF_PYTHON_CELL___";
>>     
>>         // Base64 encode code payload to bypass all quote & newline syntax errors
>>          String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));
>>      
>>          // Guarantee marker printing using try...finally and display trailing expressions using ast
>>         String payload = String.format(
>>              "import base64, ast, traceback\n" +
>>             "try:\n" +
>>  

In [ ]:
%%jshell

runPy("""
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path ; from IPython.display import display, HTML ; from IPython.display import display, HTML
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists(): os.system('git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming');

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_2'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')
""");


->  Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0] on linux
Type "help", "copyright", "credits" or "license" for more information.
>>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Cloning into '/content/BITS_programming'...
Updating files:   9% (141/1417)
Updating files:  10% (142/1417)
Updating files:  11% (156/1417)
Updating files:  11% (164/1417)
Updating files:  12% (171/1417)
Updating files:  13% (185/1417)
Updating files:  14% (199/1417)
Updating files:  15% (213/1417)
Updating files:  16% (227/1417)
Updating files:  17% (241/1417)
Updating files:  18% (256/1417)
Updating files:  19% (270/1417)
Updating files:  20% (284/1417)
Updating files:  21% (298/1417)
Updating files:  22% (312/1417)
Updating files:  23% (326/1417)
Updating files:  24% (341/1417)
Updating files:  25% (355/1417)
Updating files:  26% (369/1417)
Updating files:  27% (383/1417)
Updating files:  28% (397/1417)
Updating files:  29% (411/1417)
Updating files:  30% (426/1417)
Updating files

In [4]:
%%jshell

runPy("""
# Download the dataset file to local storage (from publicly shared drive file)
# Original source: https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques
# (The rest of the notebook expects the original housing.csv to be present in the current directory)
os.system('gdown 1Yhw8tBCmjYrDZ5Rm1xPrP2y5bmBmxoEP');
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Downloading...
From: https://drive.google.com/uc?id=1Yhw8tBCmjYrDZ5Rm1xPrP2y5bmBmxoEP
To: /content/BITS_programming/assignments/assignment_2/housing.csv

  0%|                                               | 0.00/461k [00:00<?, ?B/s]
100%|███████████████████████████████████████| 461k/461k [00:00<00:00, 11.4MB/s]
0


In [5]:
%%jshell

runPy("""
# Load the dataset
import pandas as pd
DATASET_PATH = str(NOTEBOOK_DIR / "housing.csv")
df = pd.read_csv(DATASET_PATH)
""");


In [6]:
%%jshell

runPy("""
# Display all the number of nan values in each column and type
column_stats_raw = []
for column_name in df.columns:
  column_stats_raw.append((column_name, df[column_name].dtype, df[column_name].isna().sum()))
column_stats_df = pd.DataFrame(column_stats_raw, columns=['Column name', 'Type', 'Number of null values'])

print(column_stats_df.to_string())
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ...       Column name     Type  Number of null values
0              Id    int64                      0
1      MSSubClass    int64                      0
2        MSZoning   object                      0
3     LotFrontage  float64                    259
4         LotArea    int64                      0
5          Street   object                      0
6           Alley   object                   1369
7        LotShape   object                      0
8     LandContour   object                      0
9       Utilities   object                      0
10      LotConfig   object                      0
11      LandSlope   object                      0
12   Neighborhood   object                      0
13     Condition1   object                      0
14     Condition2   object                      0
15       BldgType   object                      0
16     HouseStyle   object                      0
17    OverallQual    int64      

In [7]:
%%jshell

runPy("""
# Deletion of rows with missing data

# Scenario: We are trying to test the correlation between LotFrontage and SalePrice,
# therefore we remove the rows that have NA values for them
df.dropna(subset=["LotFrontage", "SalePrice"])
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ...         Id  MSSubClass MSZoning  ...  SaleType  SaleCondition SalePrice
0        1          60       RL  ...        WD         Normal    208500
1        2          20       RL  ...        WD         Normal    181500
2        3          60       RL  ...        WD         Normal    223500
3        4          70       RL  ...        WD        Abnorml    140000
4        5          60       RL  ...        WD         Normal    250000
...    ...         ...      ...  ...       ...            ...       ...
1455  1456          60       RL  ...        WD         Normal    175000
1456  1457          20       RL  ...        WD         Normal    210000
1457  1458          70       RL  ...        WD         Normal    266500
1458  1459          20       RL  ...        WD         Normal    142125
1459  1460          20       RL  ...        WD         Normal    147500

[1201 rows x 81 columns]


In [8]:
%%jshell

runPy("""
# Mean/Median Imputation

# We are considering LotFrontage again as it is in float (numerical) and has null values

meanLotFrontage = df["LotFrontage"].mean()
medianLotFrontage = df["LotFrontage"].median()

meanImputedLotFrontage = df["LotFrontage"].fillna(meanLotFrontage)
medianImputedLotFrontage = df["LotFrontage"].fillna(medianLotFrontage)

# Showing all values might not be the best option here given the size,
# therefore using mean/median to show that math is maintained

print("Mean before performing mean imputation:", meanLotFrontage)
print("Mean after performing mean imputation:", meanImputedLotFrontage.mean())
print()
print("Median before performing median imputation:", medianLotFrontage)
print("Median after performing median imputation:", medianImputedLotFrontage.median())
print()
print("Both are almost the same/maintained because of the kind of imputation performed here.")
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Mean before performing mean imputation: 70.04995836802665
Mean after performing mean imputation: 70.04995836802664

Median before performing median imputation: 69.0
Median after performing median imputation: 69.0

Both are almost the same/maintained because of the kind of imputation performed here.


In [9]:
%%jshell

runPy("""
# Mode imputation
# (Can be considered in cases of non-numeric values where mean/median averaging isn't possible)

print("Unique values of BsmtQual/Basement Quality column:", set(df["BsmtQual"]))

# Multiple modes can exist if max count is same, therefore considering only the first one for imputation
modeBsmtQual = df["BsmtQual"].mode()[0]

print("Most repeated value of BsmtQual column:", modeBsmtQual)

print()
print("Current number of NaN values in BsmtQual:", df["BsmtQual"].isna().sum())
modeImputedBsmtQual = df["BsmtQual"].fillna(modeBsmtQual)
print("After imputation, the number of NaN values now are,", modeImputedBsmtQual.isna().sum())
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Unique values of BsmtQual/Basement Quality column: {'Gd', 'Ex', 'TA', nan, 'Fa'}
Most repeated value of BsmtQual column: TA

Current number of NaN values in BsmtQual: 37
After imputation, the number of NaN values now are, 0


In [10]:
%%jshell

runPy("""
# Arbitrary value imputation (remaining)
# (Value chosen by analyst with his domain knowledge or their understanding of the dataset)

# For example, we could choose

arbitary_value = 21.0

print("Mean of LotFrontage before arbitrary value imputation:", df["LotFrontage"].mean())
arbitaryValueImputedCol = df["LotFrontage"].fillna(arbitary_value)
print("Mean of LotFrontage after arbitrary value imputation: ", arbitaryValueImputedCol.mean())
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Mean of LotFrontage before arbitrary value imputation: 70.04995836802665
Mean of LotFrontage after arbitrary value imputation:  61.3486301369863


In [11]:
%%jshell

runPy("""
# End of tail Imputation
minLotFrontage = df["LotFrontage"].min()
maxLotFrontage = df["LotFrontage"].max()

chosenTail = minLotFrontage

print("Mean before end of tail imputation:", df["LotFrontage"].mean())
tailImputedLotFrontage = df["LotFrontage"].fillna(chosenTail)
print("Mean after end of tail imputation:", tailImputedLotFrontage.mean())
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Mean before end of tail imputation: 70.04995836802665
Mean after end of tail imputation: 61.3486301369863


In [12]:
%%jshell

runPy("""
# Random sample imputation (runtime error with logic)
import random

print("NaN values in FireplaceQu before sampling imputation:", df["FireplaceQu"].isna().sum())

# Random sample Imputation (on Fireplace Quality column)
random_sample_size = 10
random_sample = df["FireplaceQu"].dropna().sample(n=random_sample_size)

impute_random_sample = lambda x: random_sample.sample().iloc[0] if pd.isnull(x) else x

# Apply the lambda function to impute missing values in the "FireplaceQu" column
random_sample_imputed_col = df["FireplaceQu"].apply(impute_random_sample)

print("NaN values in FireplaceQu after sampling imputation: ", random_sample_imputed_col.isna().sum())
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... NaN values in FireplaceQu before sampling imputation: 690
NaN values in FireplaceQu after sampling imputation:  0


In [13]:
%%jshell

runPy("""
#	Frequent category Imputation (same as mode imputation)
print("Unique values of GarageType column:", set(df["GarageType"]))

# Multiple modes can exist if max count is same, therefore considering only the first one for imputation
modeGarageType = df["GarageType"].mode()[0]

print("Most repeated value of GarageType column:", modeGarageType)

print()
print("Current number of NaN values in BsmtQual:", df["GarageType"].isna().sum())
freqCatImputatedCol = df["GarageType"].fillna(modeGarageType)
print("After imputation, the number of NaN values now are:", freqCatImputatedCol.isna().sum())
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Unique values of GarageType column: {'Detchd', 'Basment', 'BuiltIn', '2Types', 'CarPort', nan, 'Attchd'}
Most repeated value of GarageType column: Attchd

Current number of NaN values in BsmtQual: 81
After imputation, the number of NaN values now are: 0


In [14]:
%%jshell

runPy("""
# Adding a new category as "missing"
# (Replacing NaN values with explicit category such as "missing" as a common category for them)

missing_category = "Unknown"

print("Set of unique values in GarageType col before given imputation:", set(df["GarageType"]))
imputedColumn = df["GarageType"].fillna(missing_category)
print("Set of unique values in GarageType col after given imputation:", set(imputedColumn))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Set of unique values in GarageType col before given imputation: {'Detchd', 'Basment', 'BuiltIn', '2Types', 'CarPort', nan, 'Attchd'}
Set of unique values in GarageType col after given imputation: {'Detchd', 'Basment', 'Unknown', 'BuiltIn', '2Types', 'CarPort', 'Attchd'}


In [15]:
%%jshell

runPy("""
# Random sample imputation (done earlier/repetition)
""");


In [16]:
%%jshell

runPy("""
# Regression
# We'll be using SalePrice (x) to predict LotFrontage (y) for missing/nan values
from sklearn.linear_model import LinearRegression

model = LinearRegression()
imputed_df = df.dropna(subset=["LotFrontage"])[["LotFrontage", "SalePrice"]]
X = imputed_df.SalePrice.values.reshape(len(imputed_df), 1)
Y = imputed_df.LotFrontage.values.reshape(len(imputed_df), 1)
model.fit(X, Y)

regression_imputed_lot_frontage = df[["LotFrontage", "SalePrice"]].apply(lambda x: model.predict([[x["SalePrice"]]])[0][0] if pd.isnull(x["LotFrontage"]) else x["LotFrontage"], axis=1)

print("Mean before regression imputation of LotFrontage:", df["LotFrontage"].mean())
print("Number NA values before imputation:", df["LotFrontage"].isna().sum())
print()
print("Mean after regression imputation of LotFrontage:", regression_imputed_lot_frontage.mean())
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Mean before regression imputation of LotFrontage: 70.04995836802665
Number NA values before imputation: 259

Mean after regression imputation of LotFrontage: 70.065399342723


In [17]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Current Time in IST: 2026-09-28 15:28:32
